# YOLO Detection Training Platform - Colab/Kaggle

Notebook ini menjalankan platform web training YOLO dengan cara clone dari Git.

**Langkah:**
1. Push project ini ke GitHub
2. Ganti `GIT_REPO_URL` di cell berikut dengan URL repo Anda
3. Daftar gratis di [ngrok.com](https://ngrok.com), ambil authtoken, lalu ganti `NGROK_AUTHTOKEN` di cell port forwarding
4. Jalankan semua cell secara berurutan

In [ ]:
!git clone https://github.com/Herutriana44/AI_Detection_trainer.git
%cd AI_Detection_trainer

In [ ]:
!pip install -q pyngrok
!pip install -r requirements.txt -q

In [ ]:
# Deteksi environment & path (PYTHONUNBUFFERED = log training langsung ke output)
import os
os.environ.setdefault("PYTHONUNBUFFERED", "1")
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PROJECT_DIR = os.getcwd()  # Setelah %cd di cell sebelumnya
print(f"Environment: {'Colab' if IN_COLAB else 'Kaggle' if os.path.exists('/kaggle') else 'Local'}")
print(f"Project dir: {PROJECT_DIR}")

In [ ]:
NGROK_AUTHTOKEN = "" #@param {type:"string"}

In [ ]:
# Port forwarding dengan ngrok - jalankan setelah app siap
import threading
import time

# Setup logger agar tampil di terminal & file data/logs/app.log
from utils.logger import setup_logger, get_log_path
log = setup_logger("colab")

def run_flask_app():
    from app import socketio, app
    log.info("Flask app starting on 0.0.0.0:5000")
    socketio.run(app, host="0.0.0.0", port=5000, debug=False)

# Jalankan Flask di background
log.info("Menjalankan Flask app di port 5000...")
thread = threading.Thread(target=run_flask_app, daemon=True)
thread.start()
time.sleep(3)  # Tunggu app siap

# Setup ngrok tunnel
if NGROK_AUTHTOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    public_url = ngrok.connect(5000)
    log.info("Platform siap! Buka di browser: %s", public_url)
    print(f"\n✅ Platform siap! Buka di browser:\n   {public_url}")
else:
    log.warning("NGROK_AUTHTOKEN kosong - akses terbatas")
    print("\n⚠️ NGROK_AUTHTOKEN kosong. Daftar di ngrok.com, ambil authtoken, lalu isi di cell sebelumnya.")
    print("   Tanpa ngrok, akses hanya via Colab: https://localhost:5000 (atau URL Colab proxy)")

print(f"\n📋 Log file: {get_log_path()} | Jalankan cell berikut untuk melihat log live.")

### 📋 Tampilkan Log Aplikasi

Log ditulis ke **terminal** (output cell) dan **file** `data/logs/app.log`.  
Jalankan cell di bawah untuk melihat log terbaru atau live.

In [ ]:
# Tampilkan log terbaru (50 baris terakhir)
from utils.logger import get_log_path
from pathlib import Path

log_path = get_log_path()
if log_path.exists():
    lines = log_path.read_text(encoding="utf-8").strip().split("\n")
    last_n = 50
    display_lines = lines[-last_n:] if len(lines) > last_n else lines
    print("=" * 60 + " LOG TERBARU " + "=" * 60)
    for line in display_lines:
        print(line)
    print("=" * 60 + f" ({len(display_lines)} baris) " + "=" * 60)
else:
    print("Log file belum ada. Jalankan cell Flask/ngrok terlebih dahulu.")

In [ ]:
# Live log viewer - tampilkan log secara real-time (interrupt cell untuk berhenti)
from utils.logger import get_log_path
from IPython.display import clear_output
import time

log_path = get_log_path()
last_size = 0
try:
    while True:
        if log_path.exists():
            with open(log_path, "r", encoding="utf-8") as f:
                content = f.read()
            if len(content) != last_size:
                clear_output(wait=True)
                lines = content.strip().split("\n")
                print("\n".join(lines[-80:]))  # 80 baris terakhir
                last_size = len(content)
        time.sleep(2)
except KeyboardInterrupt:
    print("\n[Log viewer dihentikan]")